# TAMP-OS — Batch GUI
**Select multiple microscopy images at once and convert them all to STL lithographs with a single click.**

> ⚠️ This notebook launches a GUI window. Run it in **Jupyter Lab / Jupyter Notebook** (not VS Code's notebook viewer).

### Install dependencies (run once)

In [ ]:
!pip install numpy pillow scipy numpy-stl

### Imports

In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
from pathlib import Path
import threading
import sys
import numpy as np
from PIL import Image
from scipy.ndimage import gaussian_filter
from stl import mesh

---
## Pipeline functions
*(same as tamp_litho.ipynb — kept here so this notebook is self-contained)*

In [ ]:
def image_to_heightmap(image_path, output_size=(512,512), invert=False,
                       blur_sigma=1.0, contrast_percentile=(2.0,98.0),
                       preserve_aspect=True, flip=True):
    img = Image.open(image_path).convert("L")
    orig_w, orig_h = img.size
    if preserve_aspect and orig_w != orig_h:
        target_w = output_size[0]
        target_h = round(target_w * orig_h / orig_w)
        actual_size = (target_w, target_h)
        print(f"[i] {orig_w}x{orig_h} -> {target_w}x{target_h}, suggested size_y={round(100*orig_h/orig_w,1)}")
    else:
        actual_size = output_size
    img = img.resize(actual_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32)
    lo = np.percentile(arr, contrast_percentile[0])
    hi = np.percentile(arr, contrast_percentile[1])
    arr = np.clip((arr - lo) / (hi - lo + 1e-8), 0.0, 1.0)
    from scipy.ndimage import gaussian_filter
    if blur_sigma > 0:
        arr = gaussian_filter(arr, sigma=blur_sigma)
    if invert:
        arr = 1.0 - arr
    if flip:
        arr = np.flipud(arr)
    return arr

def heightmap_to_stl(heightmap, output_path, base_thickness_mm=1.0,
                     max_relief_mm=3.0, physical_size_mm=(100.0,100.0)):
    rows, cols = heightmap.shape
    dx = physical_size_mm[0] / (cols - 1)
    dy = physical_size_mm[1] / (rows - 1)
    hm_ratio = cols / rows
    print_ratio = physical_size_mm[0] / physical_size_mm[1]
    if abs(hm_ratio - print_ratio) > 0.05:
        print(f"[WARNING] Aspect ratio mismatch! Consider size_y={round(physical_size_mm[0]/hm_ratio,1)}")
    z_top = base_thickness_mm + heightmap * max_relief_mm
    num_tris = (rows-1)*(cols-1)*4 + 2*((rows-1)+(cols-1))*2
    litho_mesh = mesh.Mesh(np.zeros(num_tris, dtype=mesh.Mesh.dtype))
    tri_idx = 0
    def add_tri(v0,v1,v2):
        nonlocal tri_idx
        litho_mesh.vectors[tri_idx]=[v0,v1,v2]; tri_idx+=1
    for r in range(rows-1):
        for c in range(cols-1):
            x0,y0=c*dx,r*dy; x1=(c+1)*dx; x2,y2=c*dx,(r+1)*dy; x3,y3=(c+1)*dx,(r+1)*dy
            z00,z10,z01,z11=z_top[r,c],z_top[r,c+1],z_top[r+1,c],z_top[r+1,c+1]
            add_tri([x0,y0,z00],[x1,y0,z10],[x2,y2,z01]); add_tri([x1,y0,z10],[x3,y3,z11],[x2,y2,z01])
            add_tri([x0,y0,0],[x2,y2,0],[x1,y0,0]); add_tri([x1,y0,0],[x2,y2,0],[x3,y3,0])
    xmax,ymax=(cols-1)*dx,(rows-1)*dy
    for c in range(cols-1):
        x0,x1=c*dx,(c+1)*dx; z0,z1=z_top[0,c],z_top[0,c+1]
        add_tri([x0,0,0],[x1,0,0],[x0,0,z0]); add_tri([x1,0,0],[x1,0,z1],[x0,0,z0])
    for c in range(cols-1):
        x0,x1=c*dx,(c+1)*dx; z0,z1=z_top[-1,c],z_top[-1,c+1]
        add_tri([x0,ymax,0],[x0,ymax,z0],[x1,ymax,0]); add_tri([x1,ymax,0],[x0,ymax,z0],[x1,ymax,z1])
    for r in range(rows-1):
        y0,y1=r*dy,(r+1)*dy; z0,z1=z_top[r,0],z_top[r+1,0]
        add_tri([0,y0,0],[0,y0,z0],[0,y1,0]); add_tri([0,y1,0],[0,y0,z0],[0,y1,z1])
    for r in range(rows-1):
        y0,y1=r*dy,(r+1)*dy; z0,z1=z_top[r,-1],z_top[r+1,-1]
        add_tri([xmax,y0,0],[xmax,y1,0],[xmax,y0,z0]); add_tri([xmax,y1,0],[xmax,y1,z1],[xmax,y0,z0])
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    litho_mesh.save(str(output_path))
    print(f"[OK] STL saved -> {output_path}")
    return Path(output_path)

---
## ▶ Launch the Batch GUI
Run the cell below to open the GUI window.

In [ ]:
class TAMPBatchGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("TAMP-OS Batch Lithograph Maker")
        self.root.resizable(False, False)
        self.image_paths = []
        self.running = False
        self._build_ui()

    def _build_ui(self):
        pad = dict(padx=10, pady=5)
        tk.Label(self.root, text="TAMP-OS Batch Lithograph Maker",
                 font=("Helvetica", 14, "bold")).grid(row=0, column=0, columnspan=3, pady=(14,2))
        tk.Label(self.root, text="Select microscopy images and convert them all to STL in one go.",
                 font=("Helvetica", 9), fg="gray").grid(row=1, column=0, columnspan=3, pady=(0,10))

        tk.Label(self.root, text="Images:", font=("Helvetica", 10, "bold")).grid(row=2, column=0, sticky="nw", **pad)
        frame_list = tk.Frame(self.root)
        frame_list.grid(row=2, column=1, columnspan=2, **pad)
        scrollbar = tk.Scrollbar(frame_list, orient=tk.VERTICAL)
        self.listbox = tk.Listbox(frame_list, width=50, height=6,
                                  yscrollcommand=scrollbar.set, selectmode=tk.EXTENDED)
        scrollbar.config(command=self.listbox.yview)
        self.listbox.pack(side=tk.LEFT, fill=tk.BOTH)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

        btn_frame = tk.Frame(self.root)
        btn_frame.grid(row=3, column=1, columnspan=2, sticky="w", padx=10)
        tk.Button(btn_frame, text="+ Add Images", command=self._add_images, width=14).pack(side=tk.LEFT, padx=(0,5))
        tk.Button(btn_frame, text="− Remove Selected", command=self._remove_selected, width=14).pack(side=tk.LEFT)

        tk.Label(self.root, text="Output folder:", font=("Helvetica", 10, "bold")).grid(row=4, column=0, sticky="w", **pad)
        self.output_var = tk.StringVar(value=str(Path.home() / "TAMP_output"))
        tk.Entry(self.root, textvariable=self.output_var, width=38).grid(row=4, column=1, sticky="w", **pad)
        tk.Button(self.root, text="Browse", command=self._browse_output).grid(row=4, column=2, sticky="w", padx=(0,10))

        ttk.Separator(self.root, orient="horizontal").grid(row=5, column=0, columnspan=3, sticky="ew", padx=10, pady=6)
        tk.Label(self.root, text="Parameters", font=("Helvetica", 11, "bold")).grid(row=6, column=0, columnspan=3, pady=(0,4))

        params = [
            ("Print width (mm)", "size_x", "100"),
            ("Print height (mm)", "size_y", "75", "⚠  Match your image aspect ratio, e.g. 75 for a 4:3 image"),
            ("Relief height (mm)", "relief_height", "3.0", "Max tactile height difference. Try 3.0 for a first print."),
            ("Base thickness (mm)", "base_thickness", "1.0"),
            ("Blur (sigma)", "blur", "1.2", "Gaussian smoothing. Increase for noisy images."),
            ("Resolution (px)", "resolution", "512", "Height map resolution. 256 = smaller file, 512 = more detail."),
        ]
        self.param_vars = {}
        for i, item in enumerate(params):
            label, key, default = item[0], item[1], item[2]
            hint = item[3] if len(item) > 3 else None
            tk.Label(self.root, text=label + ":").grid(row=7+i, column=0, sticky="w", padx=10, pady=2)
            var = tk.StringVar(value=default)
            self.param_vars[key] = var
            tk.Entry(self.root, textvariable=var, width=10).grid(row=7+i, column=1, sticky="w", padx=10, pady=2)
            if hint:
                tk.Label(self.root, text=hint, fg="gray", font=("Helvetica", 8)).grid(row=7+i, column=2, sticky="w", padx=(0,10), pady=2)

        chk_row = 7 + len(params)
        self.invert_var = tk.BooleanVar(value=False)
        self.flip_var = tk.BooleanVar(value=True)
        tk.Checkbutton(self.root, text="Invert relief (dark areas raised)",
                       variable=self.invert_var).grid(row=chk_row, column=0, columnspan=2, sticky="w", padx=10, pady=2)
        tk.Checkbutton(self.root, text="Flip vertically (recommended — matches image orientation)",
                       variable=self.flip_var).grid(row=chk_row+1, column=0, columnspan=3, sticky="w", padx=10, pady=2)

        ttk.Separator(self.root, orient="horizontal").grid(row=chk_row+2, column=0, columnspan=3, sticky="ew", padx=10, pady=6)
        self.progress = ttk.Progressbar(self.root, orient=tk.HORIZONTAL, length=420, mode="determinate")
        self.progress.grid(row=chk_row+3, column=0, columnspan=3, padx=10, pady=4)
        self.status_var = tk.StringVar(value="Ready.")
        tk.Label(self.root, textvariable=self.status_var, fg="gray",
                 font=("Helvetica", 9)).grid(row=chk_row+4, column=0, columnspan=3, pady=(0,4))
        self.run_btn = tk.Button(self.root, text="▶  Generate STLs",
                                 font=("Helvetica", 11, "bold"),
                                 bg="#2a7ae2", fg="white",
                                 activebackground="#1a5fbf", activeforeground="white",
                                 command=self._run, width=20, height=2)
        self.run_btn.grid(row=chk_row+5, column=0, columnspan=3, pady=(4,14))

    def _add_images(self):
        files = filedialog.askopenfilenames(
            title="Select microscopy images",
            filetypes=[("Image files", "*.png *.jpg *.jpeg *.tif *.tiff *.bmp"), ("All files", "*.*")]
        )
        for f in files:
            if f not in self.image_paths:
                self.image_paths.append(f)
                self.listbox.insert(tk.END, Path(f).name)

    def _remove_selected(self):
        for i in reversed(list(self.listbox.curselection())):
            self.listbox.delete(i); self.image_paths.pop(i)

    def _browse_output(self):
        folder = filedialog.askdirectory(title="Select output folder")
        if folder: self.output_var.set(folder)

    def _get_params(self):
        try:
            return {
                "size_x": float(self.param_vars["size_x"].get()),
                "size_y": float(self.param_vars["size_y"].get()),
                "relief_height": float(self.param_vars["relief_height"].get()),
                "base_thickness": float(self.param_vars["base_thickness"].get()),
                "blur": float(self.param_vars["blur"].get()),
                "resolution": int(self.param_vars["resolution"].get()),
                "invert": self.invert_var.get(),
                "flip": self.flip_var.get(),
            }
        except ValueError as e:
            messagebox.showerror("Invalid parameters", f"Please check your parameter values:\n{e}")
            return None

    def _run(self):
        if not self.image_paths:
            messagebox.showwarning("No images", "Please add at least one image.")
            return
        params = self._get_params()
        if params is None or self.running: return
        self.running = True
        self.run_btn.config(state=tk.DISABLED)
        threading.Thread(target=self._process_all, args=(params,), daemon=True).start()

    def _process_all(self, params):
        output_dir = Path(self.output_var.get())
        output_dir.mkdir(parents=True, exist_ok=True)
        total = len(self.image_paths)
        errors = []
        for i, img_path in enumerate(self.image_paths):
            name = Path(img_path).stem
            self._set_status(f"Processing {i+1}/{total}: {name}...")
            self.progress["value"] = (i / total) * 100
            self.root.update_idletasks()
            try:
                hm = image_to_heightmap(img_path, output_size=(params["resolution"], params["resolution"]),
                                        invert=params["invert"], blur_sigma=params["blur"],
                                        preserve_aspect=True, flip=params["flip"])
                heightmap_to_stl(hm, output_dir / f"{name}_lithograph.stl",
                                 base_thickness_mm=params["base_thickness"],
                                 max_relief_mm=params["relief_height"],
                                 physical_size_mm=(params["size_x"], params["size_y"]))
            except Exception as e:
                errors.append(f"{name}: {e}")
        self.progress["value"] = 100
        if errors:
            self._set_status(f"Done with {len(errors)} error(s).")
            messagebox.showerror("Errors", "Some files failed:\n\n" + "\n".join(errors))
        else:
            self._set_status(f"Done! {total} STL(s) saved to {output_dir}")
            messagebox.showinfo("Done!", f"Successfully generated {total} STL file(s).\n\nSaved to:\n{output_dir}")
        self.running = False
        self.run_btn.config(state=tk.NORMAL)

    def _set_status(self, msg):
        self.status_var.set(msg)
        self.root.update_idletasks()

# Launch the GUI
root = tk.Tk()
app = TAMPBatchGUI(root)
root.mainloop()